In [2]:
!git clone https://github.com/tue-mps/eomt.git

fatal: destination path 'eomt' already exists and is not an empty directory.


In [3]:
%cd eomt

/kaggle/working/eomt


In [4]:
!pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.4/219.4 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 86.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 63.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0

In [5]:
import torch
import torchvision

print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

PyTorch Version: 2.7.0+cu126
Torchvision Version: 0.22.0+cu126
CUDA Available: True


In [6]:
!pip install wandb

In [7]:
!pip install huggingface_hub
!huggingface-cli download tue-mps/coco_panoptic_eomt_large_640 pytorch_model.bin --local-dir ./weights

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
pytorch_model.bin: 100%|████████████████████| 1.27G/1.27G [00:09<00:00, 139MB/s]
Download complete. Moving file to weights/pytorch_model.bin
weights/pytorch_model.bin


In [11]:
import wandb
wandb.login()

wandb: Currently logged in as: b23es1027 (b23es1027-indian-institute-of-technology-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [13]:
import os

# 1. Automatically find the exact location of the COCO folders in Kaggle's inputs
val_path = None
ann_path = None

for root, dirs, files in os.walk("/kaggle/input"):
    if "val2017" in dirs and val_path is None:
        val_path = os.path.join(root, "val2017")
    if "annotations" in dirs and ann_path is None:
        ann_path = os.path.join(root, "annotations")

if not val_path or not ann_path:
    print("❌ ERROR: Could not find the COCO dataset. Make sure it is attached to the notebook!")
else:
    print(f"✅ Found validation data at: {val_path}")
    print(f"✅ Found annotations at: {ann_path}")
    
    # 2. Create the proxy zip directory
    os.makedirs("/kaggle/working/coco_zips", exist_ok=True)
    
    # Create the dummy train folder
    os.system('echo "dummy" > dummy.txt && zip -q /kaggle/working/coco_zips/train2017.zip dummy.txt && rm dummy.txt')
    print("✅ Created proxy train2017.zip")
    
    # Zip the validation and annotation folders
    val_parent = os.path.dirname(val_path)
    ann_parent = os.path.dirname(ann_path)
    
    print("⏳ Zipping validation and annotation files (this will take a few seconds)...")
    os.system(f'cd "{val_parent}" && zip -r -0 -q /kaggle/working/coco_zips/val2017.zip val2017/')
    os.system(f'cd "{ann_parent}" && zip -r -0 -q /kaggle/working/coco_zips/annotations.zip annotations/')
    print("✅ Zipping complete!")
    
    # 3. Change directory and run the validation command with the CORRECT weights path
    print("🚀 Launching evaluation...")
    os.chdir("/kaggle/working/eomt")
    
    eval_cmd = """
    WANDB_MODE=disabled python3 main.py validate \
      -c configs/dinov2/coco/panoptic/eomt_large_640.yaml \
      --model.network.masked_attn_enabled False \
      --trainer.devices 1 \
      --data.batch_size 4 \
      --data.path /kaggle/working/coco_zips \
      --model.ckpt_path weights/pytorch_model.bin
    """
    os.system(eval_cmd)

✅ Found validation data at: /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017
✅ Found annotations at: /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations
✅ Created proxy train2017.zip
⏳ Zipping validation and annotation files (this will take a few seconds)...
✅ Zipping complete!
🚀 Launching evaluation...


2026-03-25 21:11:57.558474: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774473117.580544     751 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774473117.587411     751 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774473117.609227     751 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774473117.609251     751 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774473117.609254     751 computation_placer.cc:177] computation placer alr

In [14]:
# 1. Download the missing panoptic annotations directly from the official COCO servers
!wget -q --show-progress -P /kaggle/working/coco_zips/ http://images.cocodataset.org/annotations/panoptic_annotations_trainval2017.zip

# 2. Make sure we are in the right directory
%cd /kaggle/working/eomt

# 3. Relaunch the evaluation!
!WANDB_MODE=disabled python3 main.py validate \
  -c configs/dinov2/coco/panoptic/eomt_large_640.yaml \
  --model.network.masked_attn_enabled False \
  --trainer.devices 1 \
  --data.batch_size 4 \
  --data.path /kaggle/working/coco_zips \
  --model.ckpt_path weights/pytorch_model.bin

panoptic_annotation 100%[===================>] 820.85M  51.7MB/s    in 14s     
/kaggle/working/eomt
2026-03-25 21:19:12.792534: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774473552.815131     776 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774473552.821701     776 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774473552.838507     776 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774473552.838533     776 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target mo

In [15]:
%cd /kaggle/working/eomt
!echo "weights/" >> .gitignore
!echo "*.bin" >> .gitignore

/kaggle/working/eomt
